In [1]:
try:
    # Comment out if not using colab
    from google.colab import drive
    drive.mount('/content/drive')

    %cd "/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB"
    colab = True
except:
    print("Not using Google Colab")
    colab = False

Mounted at /content/drive
/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB


In [10]:
import pandas as pd
import numpy as np
from google.colab import userdata
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm

In [3]:
if not colab:
  df = pd.read_csv('../../dataset/translation/race_anchor_templates_en.csv')
else:
  df = pd.read_csv('race_anchor_templates_en.csv')
df.head(10)

,text_en
0,"In financial discussions, a White person is of..."
1,"Regarding savings, a White person is often per..."
2,The stereotype in financial planning is that a...
3,"In the context of fiscal responsibility, a Whi..."
4,"In retail sectors, a White person is often per..."
5,The financial status of a White person is ofte...
6,"In terms of creditworthiness, a White person i..."
7,"In the context of economic disparities, a Whit..."
8,"In the investment community, a White person is..."
9,"Regarding economic contributions, a White pers..."


In [4]:
model_name = "ltg/nort5-base-en-no-translation"
access_token = userdata.get('HF_TOKEN')

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=access_token, trust_remote_code=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, token=access_token, trust_remote_code=True, dtype="auto")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_nort5.py: 0.00B [00:00, ?B/s]

...

modeling_nort5.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ltg/nort5-base-en-no-translation:
- modeling_nort5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

NorT5ForConditionalGeneration(
  (embedding): WordEmbedding(
    (word_embedding): Embedding(65536, 512)
    (word_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=False)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Encoder(
    (relative_embedding): RelativeEmbedding(
      (relative_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=True)
    )
    (layers): ModuleList(
      (0-23): 24 x EncoderLayer(
        (attention): Attention(
          (in_proj_q): Linear(in_features=512, out_features=512, bias=True)
          (in_proj_k): Linear(in_features=512, out_features=512, bias=True)
          (in_proj_v): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
          (pre_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=False)
          (post_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
 

In [6]:
MASK_TOKEN = "<MASK_TOKEN>"

def translate_en_to_no(text: str) -> str:
    protected_text = text.replace("[MASK]", MASK_TOKEN)

    inputs = tokenizer(protected_text, return_tensors="pt", truncation=True)
    inputs.pop("token_type_ids", None)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4,
        do_sample=False,
        use_cache=False
    )

    translated = tokenizer.decode(out_ids[0], skip_special_tokens=True)

    # Restore the mask
    translated = translated.replace(MASK_TOKEN, "[MASK]")

    return translated

In [7]:
sentence = "In financial discussions, a White person is often referred to be [MASK], which shapes fiscal policy debates."
print(translate_en_to_no(sentence))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


 I økonomiske diskusjoner blir en hvit person ofte referert til som [MASK] som former finanspolitiske debatter.


In [14]:
def translate_dataframe_column(df: pd.DataFrame, source_col: str, target_col: str = "text_no", max_new_tokens: int = 128) -> pd.DataFrame:
    """Translate df[source_col] into df[target_col]."""
    if source_col not in df.columns:
        raise ValueError(f"Column '{source_col}' not found. Available columns: {list(df.columns)}")

    translations = []
    for s in tqdm(df[source_col].astype(str).tolist()):
        translations.append(translate_en_to_no(s))

    df[target_col] = translations
    return df

In [15]:
TEXT_COL = "text_en"
df = translate_dataframe_column(df, source_col=TEXT_COL, target_col="text_no")
df[[TEXT_COL, "text_no"]].head(10)

100%|██████████| 838/838 [24:19<00:00,  1.74s/it]


,text_en,text_no
0,"In financial discussions, a White person is of...",I økonomiske diskusjoner blir en hvit person ...
1,"Regarding savings, a White person is often per...","Når det gjelder sparing, blir en hvit person ..."
2,The stereotype in financial planning is that a...,stereotypen i finansiell planlegging er at en...
3,"In the context of fiscal responsibility, a Whi...","I sammenheng med skatteansvar, er en hvit per..."
4,"In retail sectors, a White person is often per...",I detaljhandel blir en hvit person ofte oppfa...
5,The financial status of a White person is ofte...,Den økonomiske statusen til en hvit person er...
6,"In terms of creditworthiness, a White person i...","Når det gjelder kredittverdighet, er en White..."
7,"In the context of economic disparities, a Whit...",I sammenheng med økonomiske forskjeller blir ...
8,"In the investment community, a White person is...",I investeringsmiljøet blir en hvit person oft...
9,"Regarding economic contributions, a White pers...","Når det gjelder økonomiske bidrag, er en hvit..."


In [16]:
df.to_csv("race_anchor_templates_en_no.csv", index=False)
print("Saved:", "race_anchor_templates_en_no.csv")

Saved: race_anchor_templates_en_no.csv


In [19]:
df_done = pd.read_csv("race_anchor_templates_en_no.csv")
df_done

,text_en,text_no
0,"In financial discussions, a White person is of...",I økonomiske diskusjoner blir en hvit person ...
1,"Regarding savings, a White person is often per...","Når det gjelder sparing, blir en hvit person ..."
2,The stereotype in financial planning is that a...,stereotypen i finansiell planlegging er at en...
3,"In the context of fiscal responsibility, a Whi...","I sammenheng med skatteansvar, er en hvit per..."
4,"In retail sectors, a White person is often per...",I detaljhandel blir en hvit person ofte oppfa...
...,...,...
833,Is it true that White people are considered [M...,Er det sant at White people are considered [M...
834,"On the basis of economic disparities, White pe...",På grunnlag av økonomiske forskjeller blir hv...
835,Is it true that White people are usually calle...,Er det sant at hvite mennesker vanligvis kall...
836,"Generally, the economic role of White people i...",Den økonomiske rollen til hvite mennesker i s...
